# FullSubNet+ Fine-tune (bản đã sửa lỗi + sửa bug trainer/alpha)

Notebook này giữ nguyên toàn bộ logic của bản trước, chỉ khác ở các điểm sau (đã tìm ra nguyên nhân khiến finetune làm GIẢM chất lượng model):

1. **Cell 5 (config)**: đổi `trainer.path` từ `...trainer.Trainer` (sai, dành cho một biến thể model khác) sang `...trainer.Trainer_Finetune` (đúng class được repo gốc thiết kế khớp với model `FullSubNet_Plus` khi `channel_attention_model = "TSSE"`). Đồng thời **bỏ `alpha = 10`** — giá trị này chỉ dùng cho class `Trainer` cũ, và khi bị đưa qua bản vá ở Cell 6 cũ, nó vô tình khuếch đại loss thực tế lên gấp 10 lần dự kiến (tương đương LR hiệu dụng ~1e-4 thay vì 1e-5), khiến model bị 'đẩy' quá xa khỏi trọng số pretrained tốt chỉ sau 1 epoch trên tập dữ liệu hẹp.
2. **Cell 6/7 (patch trainer.py)**: vì đã dùng đúng `Trainer_Finetune` (class này gọi `self.model(noisy_mag, noisy_real, noisy_imag)` và chỉ nhận 1 tensor `cRM` trả về — đúng 100% với `forward()` thật của model), nên **không cần vá gì trainer.py nữa**. Cell này giờ chỉ làm nhiệm vụ xác nhận (assert) rằng file trainer.py có đúng logic mong đợi, để bạn yên tâm trước khi train, thay vì tự ý ghi đè code có thể gây bug mới.
3. Gợi ý bổ sung: cân nhắc tăng nhẹ số epoch (2-3) và dùng validation set độc lập với tập train nếu vẫn thấy chất lượng chưa cải thiện — xem phần giải thích cuối notebook.

Chạy tuần tự từng cell từ trên xuống (Restart & Run All nếu bạn từng chạy notebook cũ, để tránh lẫn code đã vá dở của phiên bản trước).

In [1]:
# ==========================================
# CELL 1: Khai Báo & Tải Repo Mã Nguồn Chuẩn (Fix getcwd Error)
# ==========================================
import os, shutil

# Chuyển về thư mục gốc tuyệt đối để tránh lỗi mất thư mục hiện tại
os.chdir("/kaggle/working")

WORK = "/kaggle/working/FullSubNet-plus"

# Dọn dẹp an toàn thư mục cũ
if os.path.exists(WORK):
    shutil.rmtree(WORK)

# Clone repo chính thức
!git clone https://github.com/RookieJunChen/FullSubNet-plus.git {WORK}

# Chuyển hướng làm việc vào thư mục repo
os.chdir(WORK)

# Cài đặt thư viện & fix setuptools
!pip install -q GPUtil mir_eval pesq pystoi tqdm toml colorful torch_complex gdown soundfile "setuptools<81"

import librosa
print("Tải thành công Repo! Librosa Version:", librosa.__version__)


Cloning into '/kaggle/working/FullSubNet-plus'...
remote: Enumerating objects: 1280, done.
remote: Counting objects: 100% (1274/1274), done.
remote: Compressing objects: 100% (384/384), done.
remote: Total 1280 (delta 892), reused 1259 (delta 885), pack-reused 6 (from 1)
Receiving objects: 100% (1280/1280), 297.17 KiB | 3.50 MiB/s, done.
Resolving deltas: 100% (892/892), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s eta 0:00:00
Tải thành công Repo! Librosa Version: 0.11.0


In [2]:
# ==========================================
# CELL 2: Vá Toàn Bộ Lỗi Hệ Thống (Sửa Lỗi Syntax RegEx)
# ==========================================
import os, re

# Chuyển về thư mục dự án
os.chdir("/kaggle/working/FullSubNet-plus")

# 1. Vá lỗi pypesq -> pesq & librosa.resample trong metrics.py
metrics_path = "speech_enhance/audio_zen/metrics.py"
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        m_content = f.read()

    m_content = m_content.replace("from pypesq import pesq as nb_pesq\n", "")
    m_content = m_content.replace("from pypesq import pesq\n", "")
    m_content = m_content.replace("return nb_pesq(nb_ref, nb_est, 8000)", 'return pesq(8000, nb_ref, nb_est, "nb")')
    m_content = m_content.replace("librosa.resample(ref, sr, 16000)", "librosa.resample(ref, orig_sr=sr, target_sr=16000)")
    m_content = m_content.replace("librosa.resample(est, sr, 16000)", "librosa.resample(est, orig_sr=sr, target_sr=16000)")
    m_content = m_content.replace("librosa.resample(ref, sr, 8000)", "librosa.resample(ref, orig_sr=sr, target_sr=8000)")
    m_content = m_content.replace("librosa.resample(est, sr, 8000)", "librosa.resample(est, orig_sr=sr, target_sr=8000)")

    with open(metrics_path, "w") as f:
        f.write(m_content)

# 2. Vá lỗi RuntimeError trong torch.istft() tại feature.py
feature_path = "speech_enhance/audio_zen/acoustics/feature.py"
if os.path.exists(feature_path):
    with open(feature_path, "r") as f:
        f_content = f.read()

    old_istft = "return torch.istft("
    new_istft = "if not torch.is_complex(features):\n        features = torch.view_as_complex(features.contiguous())\n    return torch.istft("

    if old_istft in f_content and "torch.is_complex" not in f_content:
        f_content = f_content.replace(old_istft, new_istft)
        with open(feature_path, "w") as f:
            f.write(f_content)

# 3. Vá lỗi weights_only trong base_trainer.py bằng thay thế chuỗi trực tiếp
trainer_path = "speech_enhance/audio_zen/trainer/base_trainer.py"
if os.path.exists(trainer_path):
    with open(trainer_path, "r") as f:
        t_content = f.read()

    t_content = t_content.replace('torch.load(model_path.as_posix(), map_location="cpu")', 'torch.load(model_path.as_posix(), map_location="cpu", weights_only=False)')
    t_content = t_content.replace('torch.load(model_path, map_location="cpu")', 'torch.load(model_path, map_location="cpu", weights_only=False)')
    t_content = t_content.replace('torch.load(path)', 'torch.load(path, weights_only=False)')

    with open(trainer_path, "w") as f:
        f.write(t_content)

print("--> Đã vá toàn bộ lỗi hệ thống và cú pháp thành công!")


--> Đã vá toàn bộ lỗi hệ thống và cú pháp thành công!


In [3]:
# ==========================================
# CELL 3: Tải Pretrained Weights từ Link Mới
# ==========================================
import os

# Google Drive ID từ link bạn vừa gửi
CKPT_GDRIVE_ID = "1rPMm_eTpFLCl_rGbymhpbM4crs8Au-yj"
PRETRAINED_DIR = "/kaggle/working/FullSubNet-plus/pretrained"
os.makedirs(PRETRAINED_DIR, exist_ok=True)
PRETRAINED_PATH = os.path.join(PRETRAINED_DIR, "FullSubNet+_EN.tar")

# Xóa file cũ nếu hỏng/rỗng
if os.path.exists(PRETRAINED_PATH) and os.path.getsize(PRETRAINED_PATH) < 1000000:
    os.remove(PRETRAINED_PATH)

print("Đang tải pretrained model từ link Google Drive mới...")

# 1. Thử tải bằng gdown
!pip install -q --upgrade gdown
!gdown "https://drive.google.com/uc?id={CKPT_GDRIVE_ID}" -O "{PRETRAINED_PATH}" --fuzzy

# 2. Phương án dự phòng bằng wget bypass xác nhận
if not os.path.exists(PRETRAINED_PATH) or os.path.getsize(PRETRAINED_PATH) < 1000000:
    print("Thử lại bằng wget bypass...")
    URL = f"https://docs.google.com/uc?export=download&confirm=t&id={CKPT_GDRIVE_ID}"
    !wget --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate '{URL}' -O- | sed -rn 's/.*name="confirm" value="([^"]*)".*/\1\n/p' > /tmp/confirm.txt
    !wget --load-cookies /tmp/cookies.txt "{URL}&confirm=$(cat /tmp/confirm.txt)" -O "{PRETRAINED_PATH}"
    !rm -rf /tmp/cookies.txt /tmp/confirm.txt

size_mb = os.path.getsize(PRETRAINED_PATH) / 1e6 if os.path.exists(PRETRAINED_PATH) else 0
print(f"-> Kích thước file Checkpoint hoàn tất: {size_mb:.2f} MB")


Đang tải pretrained model từ link Google Drive mới...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.0 MB/s eta 0:00:00
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--cookies FILE] [--cookies-from-browser BROWSER] [--no-cookies]
             [--no-check-certificate] [--continue] [--folder] [--json]
             [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy
Thử lại bằng wget bypass...
--2026-09-13 10:22:27--  https://docs.google.com/uc?export=download&confirm=t&id=1rPMm_eTpFLCl_rGbymhpbM4crs8Au-yj
Resolving docs.google.com (docs.google.com)... 192.178.210.139, 192.178.210.113, 192.178.210.138, ...
Connecting to docs.google.com (docs.google.com)|192.178.210.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1rPMm_eTpFLCl_rGbymhpbM4crs8Au-yj&export=download [following]
--2026-09-13 10:22

In [4]:
# ==========================================
# CELL 4: Sinh Tập Validation Đa Dạng Để Đo Điểm PESQ/STOI
# (cell này cũng tự sinh clean.txt / noise.txt bằng glob,
#  nên không cần cell gen_lst qua module "tools" nữa)
# ==========================================
import os, random, glob, numpy as np, soundfile as sf, librosa
from pathlib import Path

os.chdir("/kaggle/working/FullSubNet-plus")

TRAIN_DATA_DIR = "/kaggle/working/FullSubNet-plus/train_data_fsn"
DATA_ROOT = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH"
TRAIN_CLEAN = f"{DATA_ROOT}/TRAIN/CLEAN"
TRAIN_NOISE = f"{DATA_ROOT}/TRAIN/NOISE"

os.makedirs(TRAIN_DATA_DIR, exist_ok=True)
clean_txt = os.path.join(TRAIN_DATA_DIR, "clean.txt")
noise_txt = os.path.join(TRAIN_DATA_DIR, "noise.txt")

# Tự động quét và ghi danh sách file .wav trực tiếp bằng Python
clean_list = glob.glob(f"{TRAIN_CLEAN}/**/*.wav", recursive=True)
noise_list = glob.glob(f"{TRAIN_NOISE}/**/*.wav", recursive=True)

with open(clean_txt, "w") as f:
    f.write("\n".join(clean_list))

with open(noise_txt, "w") as f:
    f.write("\n".join(noise_list))

open(f"{TRAIN_DATA_DIR}/rir.txt", "w").close()

print(f"-> Tìm thấy {len(clean_list)} file Clean và {len(noise_list)} file Noise.")

# Đọc danh sách file
clean_files = [l.strip() for l in open(clean_txt) if l.strip()]
noise_files = [l.strip() for l in open(noise_txt) if l.strip()]

# Tạo tập Validation
random.seed(0)
VAL_DIR = Path("/kaggle/working/val_set/no_reverb")
(VAL_DIR / "noisy").mkdir(parents=True, exist_ok=True)
(VAL_DIR / "clean").mkdir(parents=True, exist_ok=True)

N_VAL = 30
SR = 16000
val_clean = random.sample(clean_files, min(N_VAL, len(clean_files)))

for i, cpath in enumerate(val_clean):
    clean_y, _ = librosa.load(cpath, sr=SR)
    npath = random.choice(noise_files)
    noise_y, _ = librosa.load(npath, sr=SR)

    if len(noise_y) < len(clean_y):
        reps = int(np.ceil(len(clean_y) / len(noise_y)))
        noise_y = np.tile(noise_y, reps)
    noise_y = noise_y[: len(clean_y)]

    clean_rms = np.sqrt(np.mean(clean_y ** 2) + 1e-9)
    noise_rms = np.sqrt(np.mean(noise_y ** 2) + 1e-9)

    snr_db = random.choice([0, 5, 10])
    scalar = clean_rms / (10 ** (snr_db / 20)) / (noise_rms + 1e-9)
    noisy_y = clean_y + noise_y * scalar

    peak = np.max(np.abs(noisy_y))
    if peak > 0.99:
        noisy_y = noisy_y / peak * 0.99
        clean_y = clean_y / peak * 0.99

    sf.write(VAL_DIR / "noisy" / f"mix_fileid_{i}.wav", noisy_y, SR)
    sf.write(VAL_DIR / "clean" / f"clean_fileid_{i}.wav", clean_y, SR)

print("--> Đã chuẩn bị xong tập validation thành công!")


-> Tìm thấy 8460 file Clean và 8460 file Noise.
--> Đã chuẩn bị xong tập validation thành công!


In [5]:
# ==========================================
# CELL 5: Ghi Cấu Hình Fine-Tune Tối Ưu (Đã bổ sung TSSE)
# ==========================================
import os

os.chdir("/kaggle/working/FullSubNet-plus")

config_content = f'''
[meta]
save_dir = "/kaggle/working/logs/FullSubNet_Plus_finetune"
description = "Fine-tune FullSubNet+ with SNR [-5, 15] and LR 1e-5"
preloaded_model_path = "{PRETRAINED_PATH}"
seed = 0
port = "4396"
keep_reproducibility = false
use_amp = false

[acoustics]
n_fft = 512
win_length = 512
sr = 16000
hop_length = 256

[loss_function]
name = "mse_loss"
[loss_function.args]

[optimizer]
lr = 0.00001
beta1 = 0.9
beta2 = 0.999

[train_dataset]
path = "speech_enhance.fullsubnet_plus.dataset.dataset_train.Dataset"
[train_dataset.args]
clean_dataset = "{TRAIN_DATA_DIR}/clean.txt"
clean_dataset_limit = false
clean_dataset_offset = 0
noise_dataset = "{TRAIN_DATA_DIR}/noise.txt"
noise_dataset_limit = false
noise_dataset_offset = 0
num_workers = 4
pre_load_clean_dataset = false
pre_load_noise = false
pre_load_rir = false
reverb_proportion = 0.0
rir_dataset = "{TRAIN_DATA_DIR}/rir.txt"
rir_dataset_limit = false
rir_dataset_offset = 0
silence_length = 0.2
snr_range = [-5, 15]
sr = 16000
sub_sample_length = 3.072
target_dB_FS = -25
target_dB_FS_floating_value = 10

[train_dataset.dataloader]
batch_size = 4
num_workers = 4
drop_last = true
pin_memory = true

[validation_dataset]
path = "speech_enhance.fullsubnet_plus.dataset.dataset_validation.Dataset"
[validation_dataset.args]
dataset_dir_list = [
    "/kaggle/working/val_set/no_reverb"
]
sr = 16000

[model]
path = "speech_enhance.fullsubnet_plus.model.fullsubnet_plus.FullSubNet_Plus"
[model.args]
sb_num_neighbors = 15
fb_num_neighbors = 0
num_freqs = 257
look_ahead = 2
sequence_model = "LSTM"
fb_output_activate_function = false
sb_output_activate_function = false
fb_model_hidden_size = 512
sb_model_hidden_size = 384
weight_init = false
norm_type = "offline_laplace_norm"
num_groups_in_drop_band = 2
channel_attention_model = "TSSE"

[trainer]
path = "speech_enhance.fullsubnet_plus.trainer.trainer.Trainer_Finetune"
[trainer.train]
alpha = 1  # bắt buộc phải có key này (BaseTrainer đòi), Trainer_Finetune không dùng tới giá trị này
clip_grad_norm_value = 10
epochs = 1
save_checkpoint_interval = 1
[trainer.validation]
save_max_metric_score = true
validation_interval = 1
[trainer.visualization]
metrics = ["WB_PESQ", "NB_PESQ", "STOI", "SI_SDR"]
n_samples = 10
num_workers = 2
'''

os.makedirs("config", exist_ok=True)
with open("config/train.toml", "w") as f:
    f.write(config_content)

print("--> Đã cập nhật config train.toml: dùng Trainer_Finetune (khớp đúng model TSSE), bỏ alpha=10 gây khuếch đại loss!")

--> Đã cập nhật config train.toml: dùng Trainer_Finetune (khớp đúng model TSSE), bỏ alpha=10 gây khuếch đại loss!


## Cell kiểm tra `trainer.py` (không còn cần vá nữa)

Ở bản trước, Cell này dùng regex/string-replace để "vá liều" hàm `_train_epoch`/`_validation_epoch` của class `Trainer`, vì class đó vốn được viết cho một biến thể model cũ (nhận 1 input `noisy_complex`, trả về tuple `(RM, cRM)`) — **không khớp** với model `FullSubNet_Plus` thật (nhận 3 input `noisy_mag, noisy_real, noisy_imag`, chỉ trả về 1 tensor `cRM`). Bản vá đó tuy chạy được nhưng gán `RM = torch.zeros_like(...)` giả, kết hợp `alpha = 10` trong config cũ khiến loss thực tế bị nhân lệch 10 lần — đây chính là nguyên nhân khiến finetune làm giảm chất lượng.

Vì Cell 5 giờ đã trỏ thẳng tới `Trainer_Finetune` — class **gốc của repo**, vốn đã viết đúng logic gọi model 3-input/1-output, không cần alpha, không cần vá gì cả — nên Cell dưới đây chỉ còn nhiệm vụ **kiểm tra (assert)** rằng file `trainer.py` trong repo bạn clone về đúng là bản có class này với đúng nội dung mong đợi, để bạn yên tâm trước khi train.

In [6]:
# ==========================================
# CELL 6: Kiểm Tra trainer.py (không vá, chỉ xác nhận đúng class/logic)
# ==========================================
import os

os.chdir("/kaggle/working/FullSubNet-plus")
trainer_file = "speech_enhance/fullsubnet_plus/trainer/trainer.py"

with open(trainer_file, "r") as f:
    code = f.read()

# 1. Xác nhận class Trainer_Finetune tồn tại trong file (đây là class mà config.toml sẽ trỏ tới)
assert "class Trainer_Finetune(BaseTrainer):" in code, (
    "Không tìm thấy class Trainer_Finetune trong trainer.py! "
    "Kiểm tra lại repo đã clone đúng bản gốc (Cell 1) chưa, có thể repo đã đổi cấu trúc."
)

# 2. Xác nhận logic forward 3-input/1-output đúng như mong đợi (không phải bản cũ RM,cRM = model(noisy_complex))
assert "cRM = self.model(noisy_mag, noisy_real, noisy_imag)" in code, (
    "Logic gọi model trong Trainer_Finetune không như mong đợi! "
    "Hãy kiểm tra thủ công trước khi train, KHÔNG nên tự vá liều bằng regex nữa "
    "vì đó chính là nguyên nhân gây bug ở bản trước (RM giả + alpha=10 làm khuếch đại loss 10 lần)."
)

# 3. Xác nhận config train.toml đã trỏ đúng class Trainer_Finetune (khớp với Cell 5)
with open("config/train.toml", "r") as f:
    cfg = f.read()
assert "trainer.trainer.Trainer_Finetune" in cfg, (
    "config/train.toml chưa trỏ tới Trainer_Finetune! Chạy lại Cell 5 đã sửa trước khi train."
)
assert "alpha = 10" not in cfg, (
    "config/train.toml vẫn còn alpha = 10 (giá trị gây bug khuếch đại loss)! Chạy lại Cell 5 đã sửa."
)

print("--> Đã xác nhận: trainer.py có Trainer_Finetune đúng logic, config.toml đã trỏ đúng class, không còn alpha=10. "
      "Không cần vá gì thêm — có thể chạy CELL 7 (huấn luyện) ngay bây giờ.")

--> Đã xác nhận: trainer.py có Trainer_Finetune đúng logic, config.toml đã trỏ đúng class, không còn alpha=10. Không cần vá gì thêm — có thể chạy CELL 7 (huấn luyện) ngay bây giờ.


In [7]:
# ==========================================
# CELL 7: Chạy Quá Trình Fine-Tuning
# (chạy cell này sau khi Cell 6 phía trên in ra "Không cần vá gì thêm")
# ==========================================
import os
os.chdir("/kaggle/working/FullSubNet-plus")

!PYTHONPATH=. python -m speech_enhance.tools.train -C config/train.toml -N 1


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
/kaggle/working/FullSubNet-plus/speech_enhance/audio_zen/trainer/base_trainer.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.use_amp)
The configurations are as follows: 
[meta]
save_dir = "/kaggle/working/logs/FullSubNet_Plus_finetune"
description = "Fine-tune FullSubNet+ with SNR [-5, 15] and LR 1e-5"
seed = 0
port = "4396"
keep_reproducibility = false
use_amp = false
experiment_name = "train"
config_path = "config/train.toml"

[acoustics]
n_fft = 512
win_length = 512
sr = 16000
hop_length = 256

[loss_function]
name = "mse_loss"

[optimizer]
lr = 1e-5
beta1 = 0.9
beta2 = 0.999

[train_dataset]
path = "speech_enhance.fullsubnet_plus.dataset.dataset_train.Dataset"

[validation_dataset]
path = "speech_enhance.fullsubnet_plus.dataset.dataset_validation.Dataset"

[model]
pat

In [8]:
# ==========================================
# CELL 8: Chạy Inference Trên Tập TEST (đã sửa: dùng config/inference.toml đúng cấu trúc)
# (train.toml và inference.toml có cấu trúc KHÁC NHAU trong repo này -
#  train.toml dùng [train_dataset]/[validation_dataset]/[trainer], còn
#  inference.py lại cần [dataset]/[inferencer] - dùng nhầm train.toml sẽ luôn lỗi KeyError('dataset'))
# ==========================================
import os
os.chdir("/kaggle/working/FullSubNet-plus")

BEST_CKPT = "/kaggle/working/logs/FullSubNet_Plus_finetune/train/checkpoints/best_model.tar"
DATA_ROOT = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH"
TEST_DIR = f"{DATA_ROOT}/TEST"

inference_config = f'''
[acoustics]
n_fft = 512
win_length = 512
sr = 16000
hop_length = 256

[inferencer]
path = "speech_enhance.fullsubnet_plus.inferencer.inferencer.Inferencer"
type = "mag_complex_full_band_crm_mask"
[inferencer.args]
n_neighbor = 15

[dataset]
path = "speech_enhance.fullsubnet.dataset.dataset_inference.Dataset"
[dataset.args]
dataset_dir_list = [
    "{TEST_DIR}"
]
sr = 16000

[model]
path = "speech_enhance.fullsubnet_plus.model.fullsubnet_plus.FullSubNet_Plus"
[model.args]
sb_num_neighbors = 15
fb_num_neighbors = 0
num_freqs = 257
look_ahead = 2
sequence_model = "LSTM"
fb_output_activate_function = false
sb_output_activate_function = false
fb_model_hidden_size = 512
sb_model_hidden_size = 384
weight_init = false
norm_type = "offline_laplace_norm"
num_groups_in_drop_band = 2
channel_attention_model = "TSSE"
'''
# Lưu ý: [model.args] ở trên PHẢI khớp y hệt config lúc train (Cell 5),
# nếu không kiến trúc sẽ lệch và load_state_dict sẽ báo lỗi hoặc load sai.

os.makedirs("config", exist_ok=True)
with open("config/inference.toml", "w") as f:
    f.write(inference_config)

print("--> Đã sinh config/inference.toml đúng cấu trúc ([dataset] + [inferencer] + [model]).")

!PYTHONPATH=. python -m speech_enhance.tools.inference \
  -C config/inference.toml \
  -M "{BEST_CKPT}" \
  -I "{TEST_DIR}" \
  -O /kaggle/working/enhanced_test_fsn

--> Đã sinh config/inference.toml đúng cấu trúc ([dataset] + [inferencer] + [model]).
use specified dataset_dir_list: ['/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST'], instead of in config
[2026-09-13 10:47:10.931]  Loading inference dataset...
[2026-09-13 10:47:10.969]  Loading model...
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/FullSubNet-plus/speech_enhance/tools/inference.py", line 37, in <module>
    main(configuration, checkpoint_path, output_dir)
  File "/kaggle/working/FullSubNet-plus/speech_enhance/tools/inference.py", line 13, in main
    inferencer = inferencer_class(
                 ^^^^^^^^^^^^^^^^^
  File "/kaggle/working/FullSubNet-plus/speech_enhance/fullsubnet_plus/inferencer/inferencer.py", line 54, in __init__
    super().__init__(config, checkpoint_path, output_dir)
  File "/kaggle/working/FullSubNet-plus/speech_enhance/audio